# import

In [111]:
import sys
sys.path.append('..')
import numpy as np
from collections import defaultdict
from StatTools.generators.ndfnoise_generator import ndfnoise
from StatTools.visualization import plot_ff
from StatTools.analysis.dfa import DFA, dfa, dfa_worker
from StatTools.analysis.utils import(
    analyse_zero_cross_ff,
    analyse_cross_ff_linregress,
    ff_params,
    var_estimation
) 
from scipy.stats import linregress
from synth import postprocess_collisions, gen_traj, draw_ants_with_direction, draw_traj
from joblib import Parallel, delayed, cpu_count
from tqdm import tqdm
import matplotlib.pyplot as plt
import pickle
import pandas as pd
import seaborn as sns
import plotly.express as px
from pathlib import Path

# dfa estimate real ants H

In [112]:
def worker(series, n_integral, s_values):
        _, F2_s = dfa(dataset=series,
                    degree=2,
                    s_values=s_values,
                    processes=1,
                    n_integral=n_integral)
        F_s = np.sqrt(F2_s)
        return F_s, s_values

def plot_ff_slopes(
        hs: np.ndarray | None,
        S: np.ndarray,
        ff_parameter: ff_params,
        residuals: np.ndarray | None = None,
        ax=None,
        label:str = None,
        title:str = None,
        linestyle=None,
        color = None
        ):

    hurst = ff_parameter.slopes[0].value
    b = ff_parameter.intercept.value
    if hs is not None:
        hs_array = np.asarray(hs)
        S_new = np.repeat(S[:, np.newaxis], hs_array.shape[0], 1).T
        ax.scatter(S_new, hs_array)
    fit_func = 10 ** (hurst * np.log10(S) + b)
    ax.plot(S, fit_func, label=label, linestyle=linestyle, color=color)
    ax.set_xscale("log")
    ax.set_yscale("log")
    
    if title:
        ax.set_title(title)
    
    ax.grid(which='both', linewidth=0.5)
    return ax

In [113]:
pyeensis_dict = dict()

pyeensis_dir  = Path(r'E:\asp\Ants\P.yeensis\knn_kalman')
pbar = tqdm(pyeensis_dir.glob("*.csv"), total=len(list(pyeensis_dir.glob("*.csv"))))
for path in pbar:
    
    df = pd.read_csv(path)
    trajs = df[['frame', 'track_id', 'x', 'y']].pivot(columns='track_id', index='frame', values=['x', 'y']).values.astype(np.float32)

    # ind s_values for each random walk series
    tasks = []
    for j in range(trajs.shape[1]):
        series = trajs[:, j]
        series = series[~np.isnan(series)]

        if len(series)<48: continue

        s_values = np.array([int(np.exp(step)) for step in np.arange(np.log(6), np.log(len(series)/4), 0.4)])

        tasks.append([series, s_values])


    n_tasks = len(tasks)

    results = Parallel(n_jobs=4, backend='loky', verbose=0)(
        delayed(worker)(series, 0, s_values) for series, s_values in tqdm(tasks, desc=f'DFA calc', unit='series')
    )
    
    pyeensis_dict[path.name]=results
    
with open('../data/artifacts/pyeensis_ff_dict.pkl', 'wb') as f:
    pickle.dump(pyeensis_dict, f)

100%|██████████| 45/45 [16:30<00:00, 22.02s/it]


In [114]:
frufa_dict = dict()

frufa_dir  = Path(r'E:\asp\Ants\F.rufa\knn_kalman')
pbar = tqdm(frufa_dir.glob("*.csv"), total=len(list(frufa_dir.glob("*.csv"))))
for path in pbar:
    
    df = pd.read_csv(path)
    trajs = df[['frame', 'track_id', 'x', 'y']].pivot(columns='track_id', index='frame', values=['x', 'y']).values.astype(np.float32)

    # ind s_values for each random walk series
    tasks = []
    for j in range(trajs.shape[1]):
        series = trajs[:, j]
        series = series[~np.isnan(series)]

        if len(series)<48: continue

        s_values = np.array([int(np.exp(step)) for step in np.arange(np.log(6), np.log(len(series)/4), 0.4)])

        tasks.append([series, s_values])


    n_tasks = len(tasks)

    results = Parallel(n_jobs=4, backend='loky', verbose=0)(
        delayed(worker)(series, 0, s_values) for series, s_values in tqdm(tasks, desc=f'DFA calc', unit='series')
    )
    
    frufa_dict[path.name]=results
    
with open('../data/artifacts/frufa_ff_dict.pkl', 'wb') as f:
    pickle.dump(frufa_dict, f)

100%|██████████| 21/21 [09:55<00:00, 28.34s/it]
